# 矢量拟合
这是对矢量拟合的简要介绍。其概念和算法由 Bjørn Gustavsen 和 Adam Semlyen 于 1999 年提出 [[1](#link_ref1)]。有关更多信息，请参阅矢量拟合网站 [[2](#link_ref2)]。
矢量拟合的主要应用是在电路仿真器中对有源或无源器件的原始采样频率响应进行建模。另请参阅 [API 文档](../api/vectorFitting.html) 和 [应用示例](../examples/index.html#vector-fitting) 以获取有关 scikit-rf 中实现的更多信息。

## 数学描述
矢量拟合的思想是将一组有理模型函数拟合到一组采样频率响应 $\mathbf{\underline{H}}_\mathrm{sampled}$，例如来自 [S](https://en.wikipedia.org/wiki/Scattering_parameters)、[Y](https://en.wikipedia.org/wiki/Admittance_parameters) 或 [Z](https://en.wikipedia.org/wiki/Impedance_parameters) 矩阵。模型函数 $\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 在拉普拉斯域中定义，其中 $\mathrm{\underline{s}} = \sigma + \mathrm{j} \omega$：
\begin{equation}
\mathbf{\underline{H}}(\mathrm{\underline{s}}) = \mathbf{d} + \mathrm{\underline{s}} \mathbf{e} + \sum_{k=1}^K \frac{\underline{\mathbf{c}}_{k}}{\mathrm{\underline{s}}-\underline{p}_k}
\end{equation}

对于所需的拟合，该模型函数应在其每个频率样本 $\omega_n$ 处与给定频率响应匹配：
\begin{equation}
\mathbf{\underline{H}}(\mathrm{\underline{s}} = \mathrm{j} \omega_n) \overset{!}{=} \mathbf{\underline{H}}_\mathrm{sampled}(\omega_n)
\end{equation}

通常，$\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 是一个向量，包含模型的各个复频率响应 $\underline{H}_1(\mathrm{\underline{s}})$、$\underline{H}_2(\mathrm{\underline{s}})$、...、$\underline{H}_M(\mathrm{\underline{s}})$。$\mathbf{\underline{H}}(\mathrm{\underline{s}})$ 中的所有元素共享一组共同的复极点 $\underline{p}_k$，但具有各自的复留数 $\mathbf{\underline{c}}_k$、实常数 $\mathbf{d}$ 和实比例系数 $\mathbf{e}$ 集合，因此它们也是向量：
\begin{equation}
\mathbf{\underline{p}} = \begin{pmatrix} \underline{p}_1 & \underline{p}_2 & \underline{p}_3 & \cdots & \underline{p}_K \end{pmatrix}
\end{equation}

\begin{equation}
\mathbf{\underline{c}} = \begin{pmatrix} 
\underline{c}_{1,1} & \underline{c}_{2,1} & \underline{c}_{3,1} & \cdots & \underline{c}_{K,1} \\
\underline{c}_{1,2} & \underline{c}_{2,2} & \underline{c}_{3,2} & \cdots & \underline{c}_{K,2} \\
\vdots\\
\underline{c}_{1,M} & \underline{c}_{2,M} & \underline{c}_{3,M} & \cdots & \underline{c}_{K,M} \\
\end{pmatrix}
\end{equation}

\begin{equation}
\mathbf{d} = \begin{pmatrix} d_1 \\ d_2 \\ \vdots \\ d_M \end{pmatrix}
\end{equation}

\begin{equation}
\mathbf{e} = \begin{pmatrix} e_1 \\ e_2 \\ \vdots \\ e_M \end{pmatrix}
\end{equation}

所需拟合的极点数 $K$ 取决于响应的形状。

例如，目标可能是将 2 端口（$M = 4$）在 $N$ 个频率 $\omega_n$ 处采样的 S 矩阵的有理模型函数进行拟合：

\begin{equation}
\begin{pmatrix} 
\underline{S}_{11}(\omega_1) \\
\underline{S}_{12}(\omega_1) \\
\underline{S}_{21}(\omega_1) \\
\underline{S}_{22}(\omega_1) \\
\vdots \\
\underline{S}_{11}(\omega_\mathrm{N}) \\
\underline{S}_{12}(\omega_\mathrm{N}) \\
\underline{S}_{21}(\omega_\mathrm{N}) \\
\underline{S}_{22}(\omega_\mathrm{N})
\end{pmatrix}
\overset{!}{=}
\begin{pmatrix}
d_{11} + \mathrm{j} \omega_1 e_{11} + \sum_{k=1}^K \frac{\underline{c}_{k,11}}{\mathrm{j} \omega_1 - \underline{p}_k}
\\
d_{12} + \mathrm{j} \omega_1 e_{12} + \sum_{k=1}^K \frac{\underline{c}_{k,12}}{\mathrm{j} \omega_1 - \underline{p}_k}
\\
d_{21} + \mathrm{j} \omega_1 e_{21} + \sum_{k=1}^K \frac{\underline{c}_{k,21}}{\mathrm{j} \omega_1 - \underline{p}_k}
\\
d_{22} + \mathrm{j} \omega_1 e_{22} + \sum_{k=1}^K \frac{\underline{c}_{k,22}}{\mathrm{j} \omega_1 - \underline{p}_k}
\\
\vdots
\\
d_{11} + \mathrm{j} \omega_\mathrm{N} e_{11} + \sum_{k=1}^K \frac{\underline{c}_{k,11}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}
\\
d_{12} + \mathrm{j} \omega_\mathrm{N} e_{12} + \sum_{k=1}^K \frac{\underline{c}_{k,12}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}
\\
d_{21} + \mathrm{j} \omega_\mathrm{N} e_{21} + \sum_{k=1}^K \frac{\underline{c}_{k,21}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}
\\
d_{22} + \mathrm{j} \omega_\mathrm{N} e_{22} + \sum_{k=1}^K \frac{\underline{c}_{k,22}}{\mathrm{j} \omega_\mathrm{N} - \underline{p}_k}
\end{pmatrix}
\end{equation}

在矢量拟合过程中，模型参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$ 将以迭代方式优化，直到获得良好的拟合。

## 等效电路
矢量拟合采样频率响应的好处是可以用等效电路轻松表示有理基函数。详细推导可在 [[3](#link_ref3)] 和 [[4](#link_ref4)] 中找到。

### 常数和比例项
常数和比例项 $\mathbf{d} + \mathrm{\underline{s}} \mathbf{e}$ 可以用等效阻抗 $\underline{Z}_\mathrm{RL}(\mathrm{\underline{s}})$ 或等效导纳 $\underline{Y}_\mathrm{RC}(\mathrm{\underline{s}})$ 在电路中表示，由串联 RL 或并联 RC 电路构建。

<center><img src='./figures/circuit_series_RL.svg' width='170px'></center>

<center><img src='./figures/circuit_parallel_RC.svg' width='150px'></center>

常数和比例项的目标响应：
\begin{equation}
\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = d_i + \mathrm{\underline{s}} e_i
\end{equation}

#### 串联 RL 电路的阻抗：
\begin{equation}
\underline{Z}_\mathrm{RL}(\mathrm{\underline{s}}) = R + \mathrm{\underline{s}} L
\end{equation}
如果 $R = d_i$ 且 $L = e_i$，则此阻抗与目标响应匹配。

#### 并联 RC 电路的导纳：
\begin{equation}
\underline{Y}_\mathrm{RC}(\mathrm{\underline{s}}) = \frac{1}{R} + \mathrm{\underline{s}} C
\end{equation}
如果 $R = \frac{1}{d_i}$ 且 $C = e_i$，则此导纳与目标响应匹配。

### 实极点和留数
具有实极点 $p_{k,i}$ 和实留数 $c_{k,i}$ 的拟合的单个项 $\frac{c_{k,i}}{\mathrm{\underline{s}} - p_{k,i}}$ 可以用等效阻抗 $\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}})$ 或等效导纳 $\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}})$ 在电路中表示，由并联 RC 或串联 RL 电路构建。

实极点-留数项的目标响应：
\begin{equation}
\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = \frac{c_{k,i}}{\mathrm{\underline{s}} - p_{k,i}}
\end{equation}

#### 并联 RC 电路的阻抗：
并联 RC 电路与上述常数和比例项的电路相同，但这次使用其阻抗 $\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}})$ 而不是其导纳：
\begin{equation}
\underline{Z}_\mathrm{RC}(\mathrm{\underline{s}}) = \frac{\frac{1}{C}}{\mathrm{\underline{s}} + \frac{1}{RC}}
\end{equation}
如果 $C = \frac{1}{c_{k,i}}$ 且 $R = \frac{c_{k,i}}{-p_{k,i}}$，则此阻抗与目标响应匹配。

#### 串联 RL 电路的导纳：
串联 RL 电路与上述常数和比例项的电路相同，但这次使用其导纳 $\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}})$ 而不是其阻抗：
\begin{equation}
\underline{Y}_\mathrm{RL}(\mathrm{\underline{s}}) = \frac{\frac{1}{L}}{\mathrm{\underline{s}} + \frac{R}{L}}
\end{equation}
如果 $L = \frac{1}{c_{k,i}}$ 且 $R = \frac{-p_{k,i}}{c_{k,i}}$，则此导纳与目标响应匹配。

### 复共轭极点和留数
在矢量拟合中，复极点 $\underline{p}_k = p'_k + \mathrm{j} p''_k$ 和留数 $\underline{c}_k = c'_k + \mathrm{j} c''_k$ 总是以复共轭对 $(\underline{p}_k, \underline{p}_k^*)$ 和 $(\underline{c}_k, \underline{c}_k^*)$ 的形式出现。因此，这种复共轭对的等效电路的目标响应为：

\begin{equation}
\underline{H}_\mathrm{target}(\mathrm{\underline{s}}) = \frac{\underline{c}_k}{\mathrm{\underline{s}} - \underline{p}_k} + \frac{\underline{c}^*_k}{\mathrm{\underline{s}} - \underline{p}^*_k} = \frac{2 c'_k \mathrm{\underline{s}} - 2 (c'_k p'_k + c''_k p''_k)}{\mathrm{\underline{s}}^2 - 2 p'_k \mathrm{\underline{s}} + |\underline{p}_k|^2}
\end{equation}

有多种方法可以用无源或有源元件构建等效电路，可以调整其尺寸以匹配此目标频率响应。[[3](#link_ref3)] 中介绍并分析了四种这样的电路，这里介绍其中两种。

#### 并联 RLC 电路的阻抗
具有与电感串联的第二电阻的并联 RLC 电路提供的阻抗 $\underline{Z}_\mathrm{RCL}(\mathrm{\underline{s}})$ 可以与目标响应匹配。

<center><img src='./figures/circuit_cc-pole_rlc-passive.svg' width='200px'></center>

\begin{equation}
\underline{Z}_\mathrm{RCL}(\mathrm{\underline{s}}) = \frac{\frac{1}{C} \mathrm{\underline{s}} + \frac{R_1}{LC}}{\mathrm{\underline{s}}^2 + (\frac{1}{R_2 C}) \mathrm{\underline{s}} + \frac{1}{LC} (1 + \frac{R_1}{R_2})}
\end{equation}

如果 $C = \frac{1}{2 c'_k}$、$L = \frac{2 c'_k}{(p''_k)^2 + (\frac{c''_k p''_k}{c'_k})^2}$、$R_1 = \frac{-2(c'_k p'_k + c''_k p''_k)}{(p''_k)^2 + (\frac{c''_k p''_k}{c'_k})^2}$ 和 $R_2 = \frac{(2 c'_k)^2}{-2(c'_kp'_k-c''_kp''_k)}$，则此阻抗与目标响应匹配。

在某些情况下，计算的电阻可能为负值，这意味着无源电路需要提供增益，这通常是不可能的。如果被建模的网络是有源网络，这并不罕见，但对于纯无源网络也可能发生。一些电路仿真器通过使用某种受控源来实现负电阻。为了完全控制行为，综合的等效电路已经在电阻并联处包含压控电流源以实现负有效电阻。

<center><img src='./figures/circuit_cc-pole_rlc-active.svg' width='200px'></center>

如果受控电流源的跨导为 $g_{T,i} = \frac{-2}{R_i}$，则与电阻 $R_i$ 并联的电流源（由电阻上的电压 $V_i$ 控制）的有效电阻 $R_{i,eff}$ 等于 $-R_i$。

这样，$R_1$ 和 $R_2$ 都可以用正值和负值实现。对于正电阻，我们只需设置 $g_{T,i} = 0$。

#### 串联 RLC 电路的导纳
与电容上电压控制的并联电流源组合的串联 RLC 电路提供的导纳 $\underline{Y}_\mathrm{RCL,I}(\mathrm{\underline{s}})$ 可以与目标响应匹配。

<center><img src='./figures/circuit_series_RCL_parallel_current.svg' width='200px'></center>

\begin{equation}
\underline{Y}_\mathrm{RCL,I}(\mathrm{\underline{s}}) = \frac{1/L \mathrm{\underline{s}} + b}{\mathrm{\underline{s}}^2 + R/L \mathrm{\underline{s}} + 1 / (LC)}
\end{equation}

如果 $L = \frac{1}{2 c'_k}$、$R = \frac{-p'_k}{ c'_k}$、$C = \frac{2 c'_k}{|\underline{p}_k|^2}$ 和 $b = -2 (c'_k p'_k + c''_k p''_k)$ 且 $g_\mathrm{m} = bLC = \frac{b}{|\underline{p}_k|^2}$，则此导纳与目标响应匹配。

## 矢量拟合 $N$ 端口的等效电路
### 情况 1：矢量拟合 S 参数
具有矢量拟合散射矩阵的 $N$ 端口的等效电路由每个端口的接口网络和传输网络组成。

对于接口网络的实现，有时也称为端口网络，可以使用电流源 ($I_\mathrm{src,i})$ 与并联电阻 $R_i$（诺顿等效），或电压源 ($V_\mathrm{src,i}$) 与串联电阻 $R_i$（戴维南等效）。求解诺顿等效接口网络的基尔霍夫电流定律，得到 $V_i = R_i (I_i + I_{src,i})$。求解戴维南等效接口网络的基尔霍夫电压定律，得到 $V_i = R_i I_i + V_{src,i}$。

将两个方程与端口 $i$ 处入射 ($a_i$) 和反射 ($b_i$) 功率波的定义进行比较，可以看到源项 $I_\mathrm{src,i}$ 和 $V_\mathrm{src,i}$ 与反射功率波 $b_i$ 成正比，其中 $R_i$ 是端口 $i$ 的参考阻抗（电阻）：

\begin{equation}
b_i = \frac{1}{2 \sqrt{R_i}} (V_i - R_i I_i) = \frac{\sqrt{R_i}}{2} I_{src,i} = \frac{1}{2 \sqrt{R_i}} V_{src,i}
\end{equation}

入射功率波 $a_i$ 由端口电压 $V_i$ 和端口电流 $I_i$ 计算，无论接口网络的类型如何：

\begin{equation}
a_i = \frac{1}{2\sqrt{R_i}} (V_i + R_i I_i)
\end{equation}

#### 使用导纳表示散射响应

下图显示了具有外部端口电压 $V_i$ 和端口电流 $I_i$ 的此类端口 $i$ 的接口和传输网络结构。矢量拟合的各个频率响应 $\underline{S}_{i,j}$ 使用基于拟合参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$ 的等效导纳 $\underline{Y}_{\mathrm{S},i,j}$ 再现，如上所述。

<center><img src='./figures/circuit_equivalent_s_admittance.svg' width='500px'></center>

这种等效电路拓扑有一些优化空间。例如，用于入射功率波的受控源（驱动并联导纳堆栈）可以差分实现，产生的电流可以在单个节点中求和，以传输回相应端口的反射功率波。这在以下原理图中显示：

<center><img src='./figures/circuit_equivalent_s_admittance_differential.png' width='500px'></center>

#### 使用阻抗表示散射响应

下图显示了具有外部端口电压 $V_i$ 和端口电流 $I_i$ 的此类端口 $i$ 的接口和传输网络结构。矢量拟合的各个频率响应 $\underline{S}_{i,j}$ 使用基于拟合参数 $\mathbf{\underline{p}}$、$\mathbf{\underline{c}}$、$\mathbf{d}$ 和 $\mathbf{e}$ 的等效阻抗 $\underline{Z}_{\mathrm{S},i,j}$ 再现，如上所述。阻抗由受控电流源驱动，产生的电流与入射功率波 $a_i$ 成正比，其中 $g_{a,i} = \frac{1}{2 \sqrt{R_i}}$ 和 $r_{a,i} = \frac{\sqrt{R_i}}{2}$。代表由单极点-留数对建模的散射响应模型一部分的各个阻抗 $Z_{j,i,k}$ 上的电压 $V_{j,i,k}$ 以增益 $g_{b,j} = \frac{\sqrt{R_i}}{2}$ 传输到端口接口处的相应压控电流源，如上所述推导。

<center><img src='./figures/circuit_equivalent_s_impedance.svg' width='600px'></center>

#### 状态空间方程的直接实现
从状态空间模型直接综合等效电路相当简单，因为电路拓扑是从[状态空间方程](https://en.wikipedia.org/wiki/State-space_representation)推导出来的：

\begin{equation}
\mathbf{\dot{x}}(t) = \mathbf{A} \mathbf{x}(t) + \mathbf{B} \mathbf{u}(t)
\end{equation}
\begin{equation}
\mathbf{y}(t) = \mathbf{C} \mathbf{x}(t) + \mathbf{D} \mathbf{u}(t) + \mathbf{E} \mathbf{\dot{u}}(t)
\end{equation}

对于散射参数，输入向量 $\mathbf{u}$ 表示入射波 $\mathbf{a}$，输出向量 $\mathbf{y}$ 表示反射波 $\mathbf{b}$。无论参数表示如何，状态向量 $\mathbf{x}$ 都是定义网络动态的内部变量。

在等效电路中，电路元件的值由状态空间矩阵 A、B、C、D（和 E）中的系数给出。由于在矢量拟合极点/留数模型的情况下这些矩阵是稀疏的，非零电路元件的数量远低于完全填充的情况，从而产生紧凑的网表。

如下原理图所示，每个端口有一个接口网络，将各个输入和状态及其各自的增益传输到输出。状态网络取决于极点类型：实极点由一个状态表示，复共轭极点对由两个状态表示。对于可选的比例项 E，每个端口需要另一个微分网络。

<center><img src='./figures/circuit_equivalent_s_statespace.svg' width='600px'></center>

受控源的各个增益如下所示，主要来自散射参数作为端口电压和电流函数的定义：
- $g_{D,i,j} = \frac{2}{\sqrt{R_{0,i}}} d_{i,j} \frac{1}{2\sqrt{R_{0,j}}}$
- $f_{D,i,j} = \frac{2}{\sqrt{R_{0,i}}} d_{i,j} \frac{\sqrt{R_{0,j}}}{2}$
- $g_{E,i,j} = \frac{2}{\sqrt{R_{0,i}}} e_{i,j}$
- $g_{C,i,j} = \frac{2}{\sqrt{R_{0,i}}} c_{i,j,k}$
- $g_{a,j} = \frac{1}{2\sqrt{R_{0,j}}}$
- $f_{a,j} = \frac{\sqrt{R_{0,j}}}{2}$

有关状态空间电路综合的更多详细信息，请参阅 S. Grivet-Talocia 和 B. Gustavsen 的 "Passive Macromodeling"
[[4](#link_ref4)]。

#### scikit-rf 中的实现
scikit-rf 中的实现在过去已经改变了几次，尝试了等效导纳、等效阻抗和直接状态空间电路综合，如上所示。目前，scikit-rf 使用状态空间方程的直接实现。比较 ngspice、Xyce 和 LTspice 中的仿真运行时间，等效网表类型之间存在很大差异。虽然性能差异的确切原因难以调查，但似乎受控源的类型对任何基于改进节点分析（MNA）的仿真都有很大影响。压控电流源通常比受控电压源性能更好。反直觉的是，对于大多数 MNA 电路求解器，受控源的数量并不总是关键的，只要控制信号已经存在于线性系统的解向量中。因此，综合等效网表的文件大小并不代表仿真器中的性能。

### 情况 2：矢量拟合 Y 参数
未实现。有关一般等效电路，请参阅 [Y 参数](https://en.wikipedia.org/wiki/Admittance_parameters)。

### 情况 3：矢量拟合 Z 参数
未实现。有关一般等效电路，请参阅 [Z 参数](https://en.wikipedia.org/wiki/Impedance_parameters)。

## 参考文献
<div id='link_ref1'></div>[1] B. Gustavsen and A. Semlyen, "Rational approximation of frequency domain responses by vector fitting," in IEEE Transactions on Power Delivery, vol. 14, no. 3, pp. 1052-1061, July 1999, DOI: [10.1109/61.772353](https://doi.org/10.1109/61.772353).

<div id='link_ref2'></div>[2] https://www.sintef.no/projectweb/vectorfitting/

<div id='link_ref3'></div>[3] G. Antonini, "SPICE equivalent circuits of frequency-domain responses," in IEEE Transactions on Electromagnetic Compatibility, vol. 45, no. 3, pp. 502-512, Aug. 2003, DOI: [10.1109/TEMC.2003.815528](https://doi.org/10.1109/TEMC.2003.815528).

<div id='link_ref4'></div>[4] "Passive Macromodeling" by S. Grivet-Talocia and B. Gustavsen, Wiley, 2016, DOI: [10.1002/9781119140931](https://doi.org/10.1002/9781119140931).